In [1]:
import os
import random
import numpy as np
import torch
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# ==========================================
# 🛑 CELL 1: DETERMINISTIC ENVIRONMENT SETUP
# ==========================================
def seed_everything(seed=42):
    """Locks all random number generators to ensure 100% reproducible results."""
    # Python & OS
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # NumPy
    np.random.seed(seed)
    
    # PyTorch (CPU & GPU)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Force cuDNN to use deterministic algorithms
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Apply the global seed
SEED = 42
seed_everything(SEED)

# Define hardware device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✅ Environment locked. Global seed set to: {SEED}")
print(f"✅ Compute device set to: {DEVICE}")

✅ Environment locked. Global seed set to: 42
✅ Compute device set to: cuda


In [2]:
import torch.nn as nn

class HybridSASRec(nn.Module):
    def __init__(self, num_users, num_items, max_seq_len=20, embed_dim=64, num_heads=2, num_layers=2, dropout=0.3):
        super(HybridSASRec, self).__init__()
        
        self.user_embed = nn.Embedding(num_users, embed_dim)
        self.item_embed = nn.Embedding(num_items + 1, embed_dim, padding_idx=0) 
        self.pos_embed = nn.Embedding(max_seq_len, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dropout=dropout, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
    def forward(self, users, seqs, target_item_idx):
        # 1. Embed the input sequence and add positional encodings
        seq_embeds = self.item_embed(seqs)
        positions = torch.arange(seqs.size(1), device=seqs.device).unsqueeze(0).expand_as(seqs)
        pos_embeds = self.pos_embed(positions)
        
        x = seq_embeds + pos_embeds
        padding_mask = (seqs == 0)
        
        # 2. Pass through Transformer & get final sequence state
        out = self.transformer(x, src_key_padding_mask=padding_mask)
        seq_reps = out[:, -1, :] 
        
        # 3. Get inherent User Embedding (acts like the MLP user state)
        u_reps = self.user_embed(users)
        
        # 4. Combine Collaborative User Rep + Sequential Rep
        combined_reps = seq_reps + u_reps
        
        # 5. Get embedding for the target item (positive or negative)
        item_reps = self.item_embed(target_item_idx)
        
        # 6. Dot product for final affinity score
        scores = (combined_reps * item_reps).sum(dim=1)
        return scores

print("SASRec architecture loaded")

SASRec architecture loaded


In [3]:
import os
import pickle
import joblib
import pandas as pd
import numpy as np
import torch
from scipy.sparse import load_npz
from sklearn.preprocessing import MinMaxScaler

class HybridRecommender:
    def __init__(self, artifact_dir, sasrec_dir, nb_dir, device='cuda'):
        print("⏳ Initializing Hybrid Engine and loading artifacts...")
        self.device = device
        
        # 1. Load Artifacts
        with open(os.path.join(artifact_dir, 'artifacts.pkl'), 'rb') as f:
            self.artifacts = pickle.load(f)
            
        self.n_books = len(self.artifacts['book_map'])
        self.reverse_book_map = {v: k for k, v in self.artifacts['book_map'].items()}
        self.book_indices = torch.arange(self.n_books, device=self.device)
        self.scaler = MinMaxScaler()
        
        # 2. Load DataFrames & Text Matrices
        self.books_df = pd.read_parquet(os.path.join(artifact_dir, 'books_clean.parquet'))
        self.tfidf_matrix = load_npz(os.path.join(artifact_dir, 'tfidf_matrix.npz'))
        
        # Setup Proficiency Labels
        level_map = {'Beginner (A2)': 0, 'Intermediate (B1)': 1, 'Advanced (B2)': 2, 'Expert (C1)': 3}
        self.books_df['proficiency_label'] = self.books_df['proficiency_level'].map(level_map)
        self.valid_nb_indices = set(self.books_df[self.books_df['proficiency_level'] != 'Unknown'].index.values)
        
        # 3. Initialize & Pre-compute Naïve Bayes
        self.nb_model = joblib.load(os.path.join(nb_dir, 'nb_proficiency_model.pkl'))
        self.nb_probs = self.nb_model.predict_proba(self.tfidf_matrix)
        
        # 4. Initialize & Load SASRec Collaborative Model
        sasrec_config = torch.load(os.path.join(sasrec_dir, 'sasrec_config_1.pt'), map_location=self.device)
        self.sasrec_model = HybridSASRec(
            num_users=sasrec_config['n_users'], 
            num_items=sasrec_config['n_books'], 
            embed_dim=sasrec_config['embed_dim'], 
            dropout=sasrec_config['dropout']
        ).to(self.device)
        
        self.sasrec_model.load_state_dict(torch.load(os.path.join(sasrec_dir, 'sasrec_bpr_weights_1.pt'), map_location=self.device))
        self.sasrec_model.eval()
        
        print("✅ Smart HybridRecommender Class compiled and loaded successfully!")

    @torch.no_grad()
    def get_sasrec_scores(self, user_idx, seq):
        """Gets Sequential scores from SASRec & scales them to [0, 1]"""
        # Pad sequence for inference
        input_seq = seq[-20:]
        padded_seq = [0] * (20 - len(input_seq)) + input_seq
        
        users_tensor = torch.tensor([user_idx], dtype=torch.long, device=self.device)
        seq_tensor = torch.tensor([padded_seq], dtype=torch.long, device=self.device)
        
        logits = self.sasrec_model(users_tensor, seq_tensor, self.book_indices).cpu().numpy().flatten()
        scaled_scores = self.scaler.fit_transform(logits.reshape(-1, 1)).flatten()
        return scaled_scores

    def get_user_preferred_level(self, seq):
        """Determines the user's dominant reading level from their history"""
        if not seq: 
            return 1 # Default to Intermediate (B1) if brand new user
            
        # Look up the proficiency labels of the books they've read
        read_levels = self.books_df.loc[seq, 'proficiency_label'].dropna()
        
        if len(read_levels) == 0:
            return 1

        # Return their most frequent reading level (0, 1, 2, or 3)
        return int(read_levels.mode()[0])

    def get_nb_scores(self, seq):
        """Content Score = Probability that the book matches the user's reading level"""
        preferred_level = self.get_user_preferred_level(seq)
        
        # Slice the probability matrix to get the scores for their specific level
        return self.nb_probs[:, preferred_level]

    def recommend(self, user_idx, alpha=0.9, top_k=10, seq=[]):
        """Fuses scores using alpha (SASRec weight) and 1-alpha (NB weight)"""
        sasrec_scores = self.get_sasrec_scores(user_idx, seq)
        nb_scores = self.get_nb_scores(seq)
        
        final_scores = np.zeros(self.n_books)
        
        for i in range(self.n_books):
            if i not in self.valid_nb_indices:
                final_scores[i] = sasrec_scores[i]
            else:
                nb_penalty = (alpha) + ((1 - alpha) * nb_scores[i])
                final_scores[i] = sasrec_scores[i] * nb_penalty
            
        if seq:
            final_scores[seq] = -float('inf')
            
        top_k_indices = np.argsort(final_scores)[::-1][:top_k]
        recommendations = [self.reverse_book_map[idx] for idx in top_k_indices]
        
        return recommendations, final_scores[top_k_indices]

print("✅ Smart HybridRecommender Class compiled successfully!")

✅ Smart HybridRecommender Class compiled successfully!


In [4]:
import os
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ==========================================
# 🤖 CELL 4: THE MASTER AGENT CLASS
# ==========================================
class BookBuddyAgent(HybridRecommender):
    def __init__(self, artifact_dir, sasrec_dir, nb_dir, device='cuda', llm_model_id="Qwen/Qwen2.5-7B-Instruct"):
        # 1. Initialize the Parent Class (Loads SASRec, Naïve Bayes, and Artifacts)
        super().__init__(artifact_dir, sasrec_dir, nb_dir, device=device)
        
        # 2. Load and Build User Reading Sequences Internally
        print("⏳ Building user reading histories for context...")
        train_df = pd.read_parquet(os.path.join(artifact_dir, 'train.parquet'))
        # Filter for positive interactions (rating >= 4) to build the sequences
        train_imp = train_df[train_df['rating'] >= 4][['user_idx', 'book_idx']]
        self.train_seqs = train_imp.sort_values('user_idx').groupby('user_idx')['book_idx'].apply(list).to_dict()
        
        self.level_map_reverse = {0: 'Beginner (A2)', 1: 'Intermediate (B1)', 2: 'Advanced (B2)', 3: 'Expert (C1)'}
        
        # 3. Initialize the LLM Engine (Stage 3)
        print(f"⏳ Loading Generative LLM: {llm_model_id} (Takes ~2 mins)...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )
        
        self.tokenizer = AutoTokenizer.from_pretrained(llm_model_id)
        self.llm = AutoModelForCausalLM.from_pretrained(
            llm_model_id,
            quantization_config=bnb_config,
            device_map="auto"
        )
        print("✅ BookBuddy Master Agent Fully Initialized!")

    def _build_context(self, seq, top_10_ids):
        """Internal method: Prepares text context for the LLM"""
        # User History Context
        history_titles = []
        
        # Dynamically identify the ID column
        id_col = self.artifacts.get('id_col', 'book_id')
        if id_col not in self.books_df.columns:
            id_col = 'goodreads_book_id' if 'goodreads_book_id' in self.books_df.columns else 'book_id'
            
        for idx in seq[-5:]:
            real_id = self.reverse_book_map[idx]
            title_matches = self.books_df.loc[self.books_df[id_col] == real_id, 'title'].values
            if len(title_matches) > 0:
                history_titles.append(title_matches[0])
                
        history_text = "\n".join([f"- {t}" for t in history_titles]) if history_titles else "No reading history available."
        
        # Candidates Context
        candidates_text = ""
        for i, book_id in enumerate(top_10_ids, 1):
            book = self.books_df[self.books_df[id_col] == book_id].iloc[0]
            candidates_text += f"{i}. Title: {book['title']} | Author: {book['authors']} | Genre: {book['genres_str']} | Level: {book['proficiency_level']}\n"
            
        return history_text, candidates_text

    def get_recommendations(self, user_idx):
        """Executes the full 3-Stage Pipeline for a given user"""
        
        # 1. Fetch user's sequence from internal dictionary
        seq = self.train_seqs.get(user_idx, [])
        
        # 2. RUN STAGE 1 & 2 (Inherited SASRec + NBC Fusion)
        top_10_ids, _ = self.recommend(user_idx=user_idx, seq=seq)
        
        # 3. Build Context & Identify Reading Level
        history_text, candidates_text = self._build_context(seq, top_10_ids)
        pref_level_idx = self.get_user_preferred_level(seq)
        user_level_name = self.level_map_reverse.get(pref_level_idx, "Unknown")
        
        # 4. Construct Generalized Reasoning Prompt
        messages = [
            {
                "role": "system", 
                "content": "You are 'BookBuddy', an expert and friendly book recommendation assistant. You analyze a reader's history and select the perfect books for them, ensuring the language complexity matches their comfort level."
            },
            {
                "role": "user", 
                "content": f"""I am looking for my next great book! My current reading proficiency is at the {user_level_name} level.

Here are the last few books I have read and enjoyed:
{history_text}

Here are 10 book candidates that match my reading level:
{candidates_text}

Task:
1. Select the BEST 3 books from the candidate list for my specific taste and reading level.
2. Output your response as a numbered list.
3. For each book, write a 1-to-2 sentence personalized justification. Reference the books I have already read to explain why this new book is a great fit for my interests and language proficiency.

Format exactly like this:
1. **[Book Title]** - [Your justification]"""
            }
        ]
        
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        
        # 5. RUN STAGE 3 (LLM Re-ranking & Generation)
        with torch.no_grad():
            outputs = self.llm.generate(
                **inputs, 
                max_new_tokens=350, 
                temperature=0.7,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        # Extract and return ONLY the generated text
        input_length = inputs['input_ids'].shape[1]
        response = self.tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
        
        return response.strip()

print("✅ BookBuddyAgent Class compiled successfully!")

✅ BookBuddyAgent Class compiled successfully!


In [5]:
import random
import torch
import gc
# ==========================================
# 🚀 CELL 5: INITIALIZE AGENT
# ==========================================

# Note: This will load the Qwen model into VRAM, so it may take ~2 minutes.
book_buddy = BookBuddyAgent(
    artifact_dir='/kaggle/input/datasets/sherrytelli/bookbuddy-preprocessed/bookbuddy_artifacts', 
    sasrec_dir='/kaggle/input/models/sherrytelli/sasrec-bpr/pytorch/default/2', 
    nb_dir='/kaggle/input/models/sherrytelli/nbc/scikitlearn/default/1'
)

⏳ Initializing Hybrid Engine and loading artifacts...
✅ Smart HybridRecommender Class compiled and loaded successfully!
⏳ Building user reading histories for context...
⏳ Loading Generative LLM: Qwen/Qwen2.5-7B-Instruct (Takes ~2 mins)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✅ BookBuddy Master Agent Fully Initialized!


In [8]:
import random

# ==========================================
# 🎲 CELL 6: GET RECOMMENDATIONS (RUN MULTIPLE TIMES)
# ==========================================

# 1. Select a Random User with a Reading History
valid_users = list(book_buddy.train_seqs.keys())
RANDOM_USER = random.choice(valid_users)
n_read = len(book_buddy.train_seqs[RANDOM_USER])

print("\n" + "="*80)
print(f"📚 BOOKBUDDY PIPELINE FOR RANDOM USER: {RANDOM_USER} ({n_read} books read)")
print("="*80)
print("⏳ Analyzing history and generating personalized recommendations...\n")

# 2. Generate the Final Explainable Recommendation
final_recommendations = book_buddy.get_recommendations(user_idx=RANDOM_USER)

# 3. Display Results
print(final_recommendations)
print("\n" + "="*80)


📚 BOOKBUDDY PIPELINE FOR RANDOM USER: 1640 (21 books read)
⏳ Analyzing history and generating personalized recommendations...

1. **To Kill a Mockingbird** - This classic novel by Harper Lee explores themes of racial injustice and moral growth, much like the complex social issues in "Columbine" and "Flash and Bones." It's accessible at your reading level and will challenge you with more advanced vocabulary and sentence structures.

2. **The Book Thief** - Written by Markus Zusak, this historical fiction novel combines elements of World War II narrative with a unique storytelling perspective, similar to "The Kite Runner." It offers a compelling story that is both emotionally engaging and appropriate for your reading level.

3. **Harry Potter and the Sorcerer's Stone** - While this book is listed as Intermediate (B1), it's often recommended for beginners due to its engaging narrative and clear language. The magical world and coming-of-age themes resonate with the adventures in "Case His